# LARA 5 in Google Colab

Dieses Notebook startet die bestehende MuJoCo-Simulation und optional Basismodell und Fine-Tuning. Für einen ersten Test lass die Modellpfade leer. Für den Modellvergleich wähle eine GPU-Laufzeit, zum Beispiel A100, und setze die Pfade in Zelle 1.

- **Direkt in Colab testen**. Lege im Schlüsselmenü das Secret `STACKLAB_WORKBENCH_TOKEN` an und aktiviere den Notebook-Zugriff. Verwende einen zufälligen Schlüssel mit mindestens 24 URL-sicheren Zeichen. Führe die Zellen 1 bis 4 aus. Die Workbench erscheint unter Zelle 4.
- **Im Team nutzen**. Starte dieses Notebook in einer benannten Colab-Sitzung mit der offiziellen Colab-CLI. Der Browserport geht per SSH auf deinen Rechner. Dort stellt ngrok bei Bedarf eine HTTPS-Adresse bereit. [Schritt-für-Schritt-Anleitung für den Team-Link](https://github.com/carlo2004/Simulation-Neura-LARA-5/blob/main/docs/colab-workbench.md#team-link-mit-ngrok)

Colab Pro mit positivem Recheneinheiten-Guthaben ist für den beschriebenen interaktiven Ablauf vorgesehen. [Colab-FAQ](https://research.google.com/colaboratory/faq.html)

## 1. Einstellungen

Die Pfade können auf Dateien in `/content` oder in Google Drive zeigen. Verwende ein Basis-GGUF und entweder ein zusammengeführtes Fine-Tuning-GGUF oder einen kompatiblen GGUF-LoRA-Adapter. Im direkten Colab-Start wird Drive bei Bedarf eingebunden.

In [ ]:
import os

MODE = os.environ.get("STACKLAB_COLAB_MODE", "preview")  # preview im Notebook, team über die CLI
BASE_GGUF = os.environ.get("STACKLAB_BASE_GGUF", "")
TUNED_GGUF = os.environ.get("STACKLAB_TUNED_GGUF", "")
LORA_GGUF = os.environ.get("STACKLAB_LORA_GGUF", "")

PORT = 8787
CONCURRENT = 1  # 2 für zwei gleichzeitig laufende Team-Tests
VERCEL_ORIGIN = "https://simulation-neura-lara-5.vercel.app"
REPO = "https://github.com/carlo2004/Simulation-Neura-LARA-5.git"
REVISION = "38ebe56f3209055f6ed995a46a08e76813d96d67"

if MODE not in ("preview", "team"):
    raise ValueError("Wähle preview oder team.")
if TUNED_GGUF and LORA_GGUF:
    raise ValueError("Setze entweder TUNED_GGUF oder LORA_GGUF.")
if bool(BASE_GGUF) != bool(TUNED_GGUF or LORA_GGUF):
    raise ValueError("Für den Vergleich braucht es ein Basismodell und ein Fine-Tuning.")
print("Modellvergleich" if BASE_GGUF else "Referenzmodus ohne Modellgewichte")

## 2. Umgebung einrichten

Die Zelle richtet die Projektumgebung und die Software für das Rendern ein. Du kannst sie nach einem Abbruch in derselben Laufzeit erneut ausführen. Bei einem Fehler zeigt die Zelle die Ausgabe des betroffenen Schritts an.

In [ ]:
from collections import deque
from ctypes.util import find_library
from pathlib import Path
import shlex
import shutil
import subprocess
import sys

SESSION = Path("/content/lara-workbench")
PROJECT = SESSION / "project"
KEY_FILE = SESSION / "runner.key"
LOG = SESSION / "runner.log"
SESSION.mkdir(parents=True, exist_ok=True, mode=0o700)
SESSION.chmod(0o700)

def run(*args, cwd=None):
    command = [str(arg) for arg in args]
    label = shlex.join(command)
    print(f"> {label}", flush=True)
    tail = deque(maxlen=12)
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors="replace") as process:
        for line in process.stdout:
            tail.append(line)
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f"Setup-Schritt fehlgeschlagen ({status}): {label}\n" + "".join(tail).strip())

model_paths = [Path(value) for value in (BASE_GGUF, TUNED_GGUF, LORA_GGUF) if value]
needs_drive = any(path.is_relative_to("/content/drive") and not path.is_file() for path in model_paths)
if needs_drive and MODE == "preview":
    from google.colab import drive
    drive.mount("/content/drive")
for path in model_paths:
    if not path.is_file() or path.suffix.lower() != ".gguf":
        raise FileNotFoundError(f"GGUF-Datei fehlt oder hat das falsche Format: {path}")
if model_paths:
    run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv")

if not PROJECT.exists():
    run("git", "clone", "--no-checkout", REPO, PROJECT)
elif subprocess.run(["git", "cat-file", "-e", f"{REVISION}^{{commit}}"], cwd=PROJECT,
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode:
    run("git", "fetch", "--no-tags", "origin", REVISION, cwd=PROJECT)
run("git", "checkout", "--detach", REVISION, cwd=PROJECT)
if not find_library("OSMesa"):
    run("apt-get", "update", "-qq")
    run("apt-get", "install", "-y", "-qq", "libosmesa6")

venv = PROJECT / ".venv"
PYTHON = venv / "bin" / "python"
has_pip = PYTHON.is_file() and subprocess.run(
    [str(PYTHON), "-m", "pip", "--version"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
).returncode == 0
if not has_pip:
    uv_executable = shutil.which("uv")
    if uv_executable:
        uv = [uv_executable]
    else:
        run(sys.executable, "-m", "pip", "install", "uv")
        uv = [sys.executable, "-m", "uv"]
    if sys.version_info[:2] in ((3, 12), (3, 13)):
        interpreter = ["--python", sys.executable]
    else:
        interpreter = ["--managed-python", "--python", "3.12"]
    # Colab's Python may lack ensurepip. Seed pip even after a partial setup.
    run(*uv, "venv", "--seed", "--allow-existing", *interpreter, venv, cwd=PROJECT)
run(PYTHON, "bootstrap.py", cwd=PROJECT)

if model_paths:
    runtime_kind = subprocess.check_output(
        [str(PYTHON), "-c",
         "from stacklab.llm.catalog import lock; print(lock('qwen9b', device='cuda')['cuda_runtime_kind'])"],
        cwd=PROJECT, text=True,
    ).strip()
    if runtime_kind == "source" and not shutil.which("cmake") and not (PYTHON.parent / "cmake").is_file():
        run(PYTHON, "-m", "pip", "install", "cmake")
    if runtime_kind == "source" and (not shutil.which("g++") or not shutil.which("make")):
        run("apt-get", "update", "-qq")
        run("apt-get", "install", "-y", "-qq", "g++", "make")

## 3. Runner starten

Auf **Runner bereit** warten. Der erste CUDA-Quellbuild kann einige Minuten dauern. Das Startprotokoll liegt unter `/content/lara-workbench/runner.log`.

In [ ]:
from contextlib import closing
import http.client
import json
import re
import secrets
import signal
import socket
import time
from urllib.parse import urlsplit

def stop_runner():
    process = globals().get("RUNNER")
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        process.wait(timeout=30)
    KEY_FILE.unlink(missing_ok=True)

if "RUNNER" in globals() and RUNNER.poll() is None:
    raise RuntimeError("Der Runner läuft bereits. Zum Neustart zuerst Zelle 5 ausführen.")
with socket.socket() as probe:
    probe.settimeout(1)
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"Port {PORT} ist bereits belegt.")

origins = [VERCEL_ORIGIN]
PREVIEW_URL = None
if MODE == "preview":
    from google.colab import output, userdata

    try:
        runner_key = userdata.get("STACKLAB_WORKBENCH_TOKEN")
    except Exception:
        raise RuntimeError("Lege das Colab-Secret STACKLAB_WORKBENCH_TOKEN an und erlaube den Notebook-Zugriff.") from None
    proxy = urlsplit(output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    if proxy.scheme != "https" or not proxy.hostname or proxy.username or proxy.password:
        raise RuntimeError("Colab hat keine gültige HTTPS-Adresse für die Vorschau geliefert.")
    PREVIEW_URL = f"https://{proxy.netloc}/"
    origins.append(PREVIEW_URL.rstrip("/"))
else:
    runner_key = secrets.token_urlsafe(24)
    descriptor = os.open(KEY_FILE, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(descriptor, "w", encoding="utf-8") as stream:
        os.fchmod(stream.fileno(), 0o600)
        stream.write(runner_key + "\n")

if not isinstance(runner_key, str) or not re.fullmatch(r"[A-Za-z0-9_-]{16,128}", runner_key):
    raise ValueError("Der Sitzungsschlüssel braucht 16 bis 128 URL-sichere Zeichen.")

command = [str(PYTHON), "tools/start_colab_workbench.py",
           "--port", str(PORT), "--concurrent", str(CONCURRENT)]
for origin in origins:
    command.extend(["--origin", origin])
for option, value in (("--base", BASE_GGUF), ("--tuned", TUNED_GGUF), ("--lora", LORA_GGUF)):
    if value:
        command.extend([option, value])

env = dict(os.environ, STACKLAB_WORKBENCH_TOKEN=runner_key, OPENBLAS_NUM_THREADS="1")
with LOG.open("w", encoding="utf-8") as stream:
    RUNNER = subprocess.Popen(command, cwd=PROJECT, env=env, stdout=stream, stderr=stream,
                              start_new_session=True)
del env
deadline = time.monotonic() + 25 * 60
try:
    while RUNNER.poll() is None and time.monotonic() < deadline:
        try:
            with closing(http.client.HTTPConnection("127.0.0.1", PORT, timeout=2)) as conn:
                conn.request("GET", "/api/catalog", headers={"Authorization": "Bearer " + runner_key})
                response = conn.getresponse()
                if response.status == 200:
                    catalog = json.loads(response.read())
                    if catalog.get("lane_prompts") is not True:
                        raise RuntimeError("Der gestartete Runner unterstützt keine getrennten Prompts.")
                    break
        except (OSError, http.client.HTTPException):
            pass
        time.sleep(1)
    else:
        raise RuntimeError(f"Der Runner wurde nicht bereit. Siehe {LOG}.")
except BaseException:
    stop_runner()
    if LOG.exists():
        print(LOG.read_text(encoding="utf-8", errors="replace")[-3000:])
    raise
finally:
    del runner_key

labels = ", ".join(model["label"] for model in catalog["models"])
print("Runner bereit · " + (labels or "Referenzmodus"))

## 4. Workbench öffnen

In der Colab-Vorschau unter **Runner verbinden** die URL leer lassen und den Sitzungsschlüssel aus Colab-Secrets eingeben. Für den ersten Test **Startkoordinaten und Python** aufklappen und **Referenz ausführen** wählen. Mit Modellgewichten sind **A starten**, **B starten** und **Beide starten** verfügbar.

In [ ]:
if MODE == "preview":
    from IPython.display import IFrame, display

    display(IFrame(src=PREVIEW_URL, width="100%", height=850))
else:
    print("Öffne den SSH-Portforward auf deinem Rechner. Der Sitzungsschlüssel liegt in "
          "/content/lara-workbench/runner.key und wird nur im privaten SSH-Terminal gelesen.")

## 5. Beenden

Zum Stoppen `STOP_RUNNER = True` setzen und nur diese Zelle ausführen. Danach die Colab-Laufzeit trennen und löschen. Bei einer Team-Sitzung zusätzlich ngrok und den SSH-Tunnel schließen und die benannte Colab-Sitzung beenden.

In [ ]:
STOP_RUNNER = False
if STOP_RUNNER:
    stop_runner()
    print("Runner gestoppt.")